# PhonePe Pulse — Bronze Raw JSON Ingestion

This notebook reads the versioned PhonePe Pulse JSON dataset from Amazon S3 and creates a managed Delta table in `workspace.bronze`.

**Bronze-layer rules:**
- Preserve the original JSON content.
- Store one row per source file.
- Record the file path, size, modification time and source Git commit.
- Do not clean or transform business fields in this layer.

In [0]:
SOURCE_COMMIT = "257bf38e3095020f9f516c4c72c10bdc348781b1"

S3_RAW_PATH = (
    "s3://phonepe-pulse-lakehouse-mohit-2026/"
    f"raw/phonepe-pulse/source_commit={SOURCE_COMMIT}/data/"
)

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
EXPECTED_FILE_COUNT = 11_184

print(f"Source commit : {SOURCE_COMMIT}")
print(f"S3 raw path   : {S3_RAW_PATH}")
print(f"Bronze table  : {BRONZE_TABLE}")
print(f"Expected files: {EXPECTED_FILE_COUNT:,}")

Source commit : 257bf38e3095020f9f516c4c72c10bdc348781b1
S3 raw path   : s3://phonepe-pulse-lakehouse-mohit-2026/raw/phonepe-pulse/source_commit=257bf38e3095020f9f516c4c72c10bdc348781b1/data/
Bronze table  : workspace.bronze.phonepe_raw_json
Expected files: 11,184


In [0]:
from pyspark.sql import functions as F

raw_files_df = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.json")
    .load(S3_RAW_PATH)
    .select(
        F.col("path").alias("source_path"),
        F.col("modificationTime").alias("source_modified_at"),
        F.col("length").alias("source_size_bytes"),
        F.decode(F.col("content"), "UTF-8").alias("raw_json"),
    )
    .withColumn("source_commit", F.lit(SOURCE_COMMIT))
    .withColumn("ingested_at_utc", F.current_timestamp())
    .withColumn("content_sha256", F.sha2(F.col("raw_json"), 256))
)

actual_file_count = raw_files_df.count()

print(f"Expected JSON files: {EXPECTED_FILE_COUNT:,}")
print(f"Actual JSON files  : {actual_file_count:,}")

if actual_file_count != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"File-count validation failed: "
        f"expected {EXPECTED_FILE_COUNT:,}, "
        f"found {actual_file_count:,}"
    )

print("File-count validation passed.")

Expected JSON files: 11,184
Actual JSON files  : 11,184
File-count validation passed.


In [0]:
(
    raw_files_df
    .repartition(8)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

spark.sql(
    f"""
    ALTER TABLE {BRONZE_TABLE}
    SET TBLPROPERTIES (
        'data_layer' = 'bronze',
        'source_system' = 'phonepe_pulse',
        'source_commit' = '{SOURCE_COMMIT}'
    )
    """
)

bronze_row_count = spark.table(BRONZE_TABLE).count()

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Rows written: {bronze_row_count:,}")

if bronze_row_count != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Bronze validation failed: expected "
        f"{EXPECTED_FILE_COUNT:,}, found {bronze_row_count:,}"
    )

print("Bronze Delta table validation passed.")

Bronze table: workspace.bronze.phonepe_raw_json
Rows written: 11,184
Bronze Delta table validation passed.


In [0]:
bronze_df = spark.table(BRONZE_TABLE)

quality_result = bronze_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("source_path").alias("unique_source_paths"),
    F.sum(
        F.when(
            F.col("raw_json").isNull()
            | (F.length(F.trim(F.col("raw_json"))) == 0),
            1,
        ).otherwise(0)
    ).alias("empty_json_rows"),
    F.sum(
        F.when(F.col("content_sha256").isNull(), 1).otherwise(0)
    ).alias("missing_checksum_rows"),
).first()

print(f"Total rows             : {quality_result['total_rows']:,}")
print(f"Unique source paths    : {quality_result['unique_source_paths']:,}")
print(f"Empty JSON rows        : {quality_result['empty_json_rows']:,}")
print(f"Missing checksum rows  : {quality_result['missing_checksum_rows']:,}")

assert quality_result["total_rows"] == EXPECTED_FILE_COUNT
assert quality_result["unique_source_paths"] == EXPECTED_FILE_COUNT
assert quality_result["empty_json_rows"] == 0
assert quality_result["missing_checksum_rows"] == 0

print("All Bronze quality checks passed.")

Total rows             : 11,184
Unique source paths    : 11,184
Empty JSON rows        : 0
Missing checksum rows  : 0
All Bronze quality checks passed.
